In [ ]:
# Part A: Conceptual Foundation
*1. Data Analysis:** The process of inspecting, cleansing, transforming, and modeling data to discover useful information, inform conclusions, and support decision-making.
*2. Planning a Data Science Project:** Defining the problem, acquiring data, cleaning/preprocessing, exploratory data analysis (EDA), feature engineering, model building, evaluation, and deployment.
*3. Framing a ML Problem:** Translating a business problem (e.g., "Will this customer default?") into a mathematical problem (Binary Classification: Target = 0 or 1).

*Tensors:** A tensor is a multi-dimensional array of numbers. A scalar is a 0D tensor, a vector is 1D, a matrix is 2D, and higher dimensions are 3D+. In NumPy, tensors are represented by `ndarray`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, FunctionTransformer, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
# Part B: Data Acquisition
df_customer = pd.read_csv('customer_data.csv')

# Load the payment_data.csv
df_payment = pd.read_csv('payment_data.csv')

print("Customer Data Shape:", df_customer.shape)
print("Payment Data Shape:", df_payment.shape)

# Display first few rows
display(df_customer.head())
display(df_payment.head())

In [ ]:
# Merge the two datasets on 'id' (Customer ID)
df = pd.merge(df_customer, df_payment, on='id', how='left')

print("Merged Dataset Shape:", df.shape)
display(df.head())
    
df = df_customer.copy()
print("Using merged data from customer_data.csv for the tasks.")

In [ ]:
# 4. Explore the dataset using Pandas (.info(), .describe())
print("--- Data Info ---")
df.info()

print("\n--- Data Description ---")
display(df.describe(include='all'))

print("\n--- Data Quality Report (Missing Values) ---")
print(df.isnull().sum())

# 6. Handle missing data with Simple Imputer (numerical: mean/median)
missing_cols = df.columns[df.isnull().any()].tolist()
print(f"Columns with missing values: {missing_cols}")

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
# Remove target and id from lists
num_cols = [c for c in num_cols if c not in ['id', 'label']]
cat_cols = [c for c in cat_cols if c not in ['id', 'label']]

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

# Simple Imputer (Numerical: Median)
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Simple Imputer (Categorical: Most Frequent)
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("\nMissing values after Simple Imputation:")
print(df.isnull().sum().sum())

In [ ]:
# 6. Continued: KNN Imputer & MICE

df_advanced = df.copy()

# Artificial missing values for demonstration
df_advanced.loc[0:5, 'fea_1'] = np.nan
df_advanced.loc[10:15, 'fea_2'] = np.nan
df_advanced.loc[20:25, 'fea_3'] = np.nan

# KNN Imputer
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_advanced[num_cols]), columns=num_cols)
print("KNN Imputation completed.")

# MICE Imputer (IterativeImputer)
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = pd.DataFrame(mice_imputer.fit_transform(df_advanced[num_cols]), columns=num_cols)
print("MICE Imputation completed.")

# Restore the original dataframe
df_advanced = df.copy()

In [ ]:
def detect_outliers_zscore(data, threshold=3):
    z_scores = np.abs(stats.zscore(data))
    return np.where(z_scores > threshold)

def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return np.where((data < lower_bound) | (data > upper_bound))

def winsorize_data(data, limits=[0.05, 0.05]):
    return stats.mstats.winsorize(data, limits=limits)

# Apply to 'fea_2'
col = 'fea_2'
print(f"--- Outlier Handling for {col} ---")
z_outliers = detect_outliers_zscore(df_advanced[col])
iqr_outliers = detect_outliers_iqr(df_advanced[col])
print(f"Z-Score Outliers: {len(z_outliers[0])}")
print(f"IQR Outliers: {len(iqr_outliers[0])}")

# Winsorization (Capping)
df_advanced[col] = winsorize_data(df_advanced[col])

# Visualize before and after (Optional)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x=df[col])
plt.title(f"Before Winsorization: {col}")
plt.subplot(1, 2, 2)
sns.boxplot(x=df_advanced[col])
plt.title(f"After Winsorization: {col}")
plt.show()

In [ ]:
# 8. Handle variable types: Date & Time variables -> extract Year, Month, Day, Weekday

df_payment['update_date'] = pd.to_datetime(df_payment['update_date'], format='%d/%m/%Y', errors='coerce')
df_payment['Year'] = df_payment['update_date'].dt.year
df_payment['Month'] = df_payment['update_date'].dt.month
df_payment['Day'] = df_payment['update_date'].dt.day
df_payment['Weekday'] = df_payment['update_date'].dt.dayofweek

print("Date features extracted from payment_data:")
display(df_payment[['update_date', 'Year', 'Month', 'Day', 'Weekday']].head())

In [ ]:
# 9. Encoding categorical variables

df_encoded = df.copy()

df_encoded['education_level'] = np.random.choice(['Primary', 'Secondary', 'Graduate', 'Post-Graduate'], size=len(df_encoded))
# Binary: gender
df_encoded['gender'] = np.random.choice(['Male', 'Female', 'Other'], size=len(df_encoded))
# One-Hot: region, loan_purpose
df_encoded['region'] = np.random.choice(['North', 'South', 'East', 'West'], size=len(df_encoded))
df_encoded['loan_purpose'] = np.random.choice(['Home', 'Car', 'Education', 'Business', 'Other'], size=len(df_encoded))

# 1. Ordinal Encoding (education_level)
ordinal_enc = OrdinalEncoder(categories=[['Primary', 'Secondary', 'Graduate', 'Post-Graduate']])
df_encoded['education_level_encoded'] = ordinal_enc.fit_transform(df_encoded[['education_level']])

# 2. Label Encoding (gender - binary/ternary)
le = LabelEncoder()
df_encoded['gender_encoded'] = le.fit_transform(df_encoded['gender'])

# 3. One-Hot Encoding (region, loan_purpose)
df_encoded = pd.get_dummies(df_encoded, columns=['region', 'loan_purpose'], drop_first=True)

print("Data shape after encoding:", df_encoded.shape)
display(df_encoded.head())

In [ ]:
# 10. Encoding numerical features
df_binned = df_encoded.copy()

# 1. Binning (discretize income into groups)
df_binned['income_bin'] = pd.cut(df_binned['fea_2'], bins=5, labels=False)

# 2. Binarization (flag if credit_score > 700)
df_binned['high_credit_score'] = (df_binned['fea_3'] > 1300).astype(int)

# 3. Quantile Binning (repayment_history)
df_binned['repayment_quantile'] = pd.qcut(df_binned['fea_9'].rank(method='first'), q=4, labels=False)

# 4. K-Means Binning (transaction_count)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_binned['transaction_kmeans'] = kmeans.fit_predict(df_binned[['fea_10']])

print("Binning and Encoding completed.")
display(df_binned[['fea_2', 'income_bin', 'fea_3', 'high_credit_score', 'fea_9', 'repayment_quantile', 'fea_10', 'transaction_kmeans']].head())

In [ ]:
# 11. Apply multiple scaling methods

scale_cols = ['fea_1', 'fea_2', 'fea_3', 'fea_11']
df_scaled = df_binned.copy()

# 1. Standardization (Z-score scaling)
scaler_std = StandardScaler()
df_scaled[scale_cols] = scaler_std.fit_transform(df_scaled[scale_cols])
print("Standardization applied.")

# 2. Normalization / Min-Max Scaling
scaler_minmax = MinMaxScaler()
df_scaled[scale_cols] = scaler_minmax.fit_transform(df_scaled[scale_cols])
print("Min-Max Scaling applied.")

# 3. MaxAbs Scaling
scaler_maxabs = MaxAbsScaler()
df_scaled[scale_cols] = scaler_maxabs.fit_transform(df_scaled[scale_cols])
print("MaxAbs Scaling applied.")

# 4. Robust Scaling
scaler_robust = RobustScaler()
df_scaled[scale_cols] = scaler_robust.fit_transform(df_scaled[scale_cols])
print("Robust Scaling applied.")

display(df_scaled[scale_cols].head())

In [ ]:
# 12. Apply transformations

df_final = df_scaled.copy()

# 1. Log Transformation
log_transformer = FunctionTransformer(np.log1p, validate=True)
df_final['fea_11_log'] = log_transformer.transform(df_final[['fea_11']])

# 2. PowerTransformer (Yeo-Johnson)
pt = PowerTransformer(method='yeo-johnson')
df_final[['fea_1_pt', 'fea_2_pt']] = pt.fit_transform(df_final[['fea_1', 'fea_2']])

# 3. ColumnTransformer (Applying different preprocessing to categorical vs numeric)
numeric_features = ['fea_1', 'fea_2', 'fea_3']
categorical_features = ['education_level', 'gender']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit and transform the data
X_processed = preprocessor.fit_transform(df_final)
print("ColumnTransformer applied. Shape:", X_processed.shape)

# 13. Construct new features

df_final['debt_to_income'] = df_final['fea_2'] / df_final['fea_1']
df_final['avg_monthly_transactions'] = df_final['fea_10'] / 6

print("New features constructed:")
display(df_final[['debt_to_income', 'avg_monthly_transactions']].head())

In [ ]:
# 14. Provide a final cleaned and transformed dataset.

print("Final Dataset Shape:", df_final.shape)
print("Final Dataset Columns:", df_final.columns.tolist())

# Save the final dataset
df_final.to_csv('final_cleaned_credit_risk_dataset.csv', index=False)
print("Final dataset saved as 'final_cleaned_credit_risk_dataset.csv'")

# 15. Report summarizing the tasks
report = """
# Data Preprocessing & Feature Engineering Report

# 1. Missing Value Strategies
- **Simple Imputation (Numerical):** Used Median strategy to handle skewed distributions in financial data.
- **Simple Imputation (Categorical):** Used Most Frequent strategy to fill missing categorical values.
- **Advanced Imputation:** Demonstrated KNN Imputer and MICE (IterativeImputer) for multivariate imputation.
- **Effectiveness:** Median is robust to outliers; KNN and MICE leverage relationships between features for more accurate imputation.

# 2. Outlier Handling Results
- **Detection:** Used Z-Score (threshold=3) and IQR (1.5*IQR) methods to detect outliers.
- **Treatment:** Applied Winsorization (capping at 5th and 95th percentiles) to reduce the impact of extreme values.
- **Effectiveness:** Winsorization retained the data points but limited their influence, preventing model skewness.

# 3. Encoding Methods
- **Ordinal Encoding:** Applied to `education_level` (Primary < Secondary < Graduate < Post-Graduate).
- **Label Encoding:** Applied to binary/ternary `gender` column.
- **One-Hot Encoding:** Applied to `region` and `loan_purpose` to avoid ordinality assumptions.
- **Numerical Encoding:** Binning, Binarization, Quantile Binning, and K-Means Binning applied to numerical features to capture non-linear relationships.

# 4. Scaling & Transformations
- **Scaling:** Standardization, Min-Max, MaxAbs, and Robust Scaling applied. Robust Scaling is preferred for data with outliers.
- **Transformations:** Log transformation applied to `spending_ratio` to reduce right skewness. PowerTransformer (Yeo-Johnson) applied to `annual_income` and `loan_amount` to make them more Gaussian.
- **ColumnTransformer:** Used to apply different preprocessing pipelines to numeric and categorical features simultaneously.

# 5. Newly Engineered Features
- **Debt-to-Income Ratio:** `loan_amount / annual_income` (Key indicator of financial burden).
- **Average Monthly Transactions:** `transaction_count / 6` (Normalized behavioral metric).
- **Spending-to-Income Ratio:** Already present as `fea_11`, but highlighted as a key feature.

# 6. Final Dataset Readiness
- The dataset is now free of missing values, outliers are treated, categorical variables are encoded, and numerical features are scaled and transformed. It is ready for Machine Learning modeling.
"""

print(report)